<h1><center></center></h1>
<h1><center>Elevvo Internship</center></h1>
<h1><center>Task 4</center></h1>
<h2><center>Named Entity Recognition (NER)</center></h2>

# **Hands on Task 4**

- We build a sequence labeling pipeline using the CoNLL‑2003 dataset (English).
- We compare a rule‑based approach and model‑based NER using two spaCy pipelines.
- We evaluate with exact‑span Precision/Recall/F1 and visualize entities with displaCy.

# **1- Data Collection**

**Setup**

In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"mohammedtaha778","key":"6a7ede5a84baa80af080f45a62197a14"}'}

**Move kaggle.json to the correct directory**

In [2]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

**Install Kaggle CLI**

In [3]:
!pip install kaggle --quiet

**Download the dataset using its Kaggle identifier**

In [4]:
!kaggle datasets download -d alaakhaled/conll003-englishversion

Dataset URL: https://www.kaggle.com/datasets/alaakhaled/conll003-englishversion
License(s): CC0-1.0
  0% 0.00/960k [00:00<?, ?B/s]
100% 960k/960k [00:00<00:00, 756MB/s]


**Unzip the downloaded dataset**

In [5]:
!unzip conll003-englishversion.zip

Archive:  conll003-englishversion.zip
  inflating: metadata                
  inflating: test.txt                
  inflating: train.txt               
  inflating: valid.txt               


**Import**

In [6]:
import pandas as pd

**Read TXT files → raw sentence blocks**

In [7]:
read = lambda p: open(p, encoding="utf-8").read().strip().split("\n\n")
train_raw, val_raw, test_raw = map(read, ["train.txt","valid.txt","test.txt"])

**Parse blocks → (tokens, ner_tags) lists**

In [8]:
parse = lambda sents: [([l.split()[0] for l in sent.splitlines() if not l.startswith("-DOCSTART-")],
                        [l.split()[-1] for l in sent.splitlines() if not l.startswith("-DOCSTART-")]) for sent in sents]
train_parsed, val_parsed, test_parsed = map(parse, [train_raw, val_raw, test_raw])

**Build DataFrames (tokens, ner_tags, joined text)**

In [9]:
to_df = lambda P: pd.DataFrame({"tokens":[t for t,_ in P],"ner_tags":[y for _,y in P],"text":[" ".join(t) for t,_ in P]})
df_train, df_val, df_test = map(to_df, [train_parsed, val_parsed, test_parsed])

**Preview**

In [10]:
df_train.head(10)[["text","ner_tags"]]

,text,ner_tags
0,,[]
1,EU rejects German call to boycott British lamb .,"[B-ORG, O, B-MISC, O, O, O, B-MISC, O, O]"
2,Peter Blackburn,"[B-PER, I-PER]"
3,BRUSSELS 1996-08-22,"[B-LOC, O]"
4,The European Commission said on Thursday it di...,"[O, B-ORG, I-ORG, O, O, O, O, O, O, B-MISC, O,..."
5,Germany 's representative to the European Unio...,"[B-LOC, O, O, O, O, B-ORG, I-ORG, O, O, O, B-P..."
6,""" We do n't support any such recommendation be...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
7,He said further scientific study was required ...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
8,He said a proposal last month by EU Farm Commi...,"[O, O, O, O, O, O, O, B-ORG, O, O, B-PER, I-PE..."
9,Fischler proposed EU-wide measures after repor...,"[B-PER, O, B-MISC, O, O, O, O, B-LOC, O, B-LOC..."


**sentences SIZE**

In [11]:
len(df_train), len(df_val), len(df_test)

(14987, 3466, 3684)

**Unique tag set**

In [12]:
labels = sorted({t for seq in pd.concat([df_train.ner_tags,df_val.ner_tags,df_test.ner_tags]) for t in seq})
labels

['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']

**Token counts**

In [13]:
toksum = lambda df: sum(len(x) for x in df.tokens)
toksum(df_train), toksum(df_val), toksum(df_test)

(203621, 51362, 46435)

**Save csv's**

In [14]:
df_train.to_csv("conll_train.csv", index=False)
df_val.to_csv("conll_val.csv", index=False)
df_test.to_csv("conll_test.csv", index=False)

# **2. Preprocessing**

**Import**

In [15]:
import re

**Normalize whitespace**

In [16]:
norm = lambda s: " ".join(str(s).split())
df_train["text"] = df_train["text"].map(norm)
df_val["text"] = df_val["text"].map(norm)
df_test["text"] = df_test["text"].map(norm)

**Drop any empty sentences**

In [17]:
f = lambda df: df[df.tokens.map(len)>0].reset_index(drop=True)
df_train, df_val, df_test = f(df_train), f(df_val), f(df_test)

**Sanity-check tag set**

In [18]:
labels = sorted({t for seq in pd.concat([df_train.ner_tags,df_val.ner_tags,df_test.ner_tags]) for t in seq})
labels

['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']

**BIO → character spans helper**

In [19]:
def bio_to_spans(tokens,tags):
    spans=[]
    text=" ".join(tokens)
    offs=[0]
    [offs.append(offs[-1]+len(w)+1) for w in tokens]
    i=0
    while i<len(tags):
        if tags[i].startswith("B-"):
            lab=tags[i][2:]
            j=i+1 # Initialize j here
            while j<len(tags) and tags[j]==f"I-{lab}":
                j+=1
            spans.append((lab, offs[i], offs[j]-1 if j<len(tokens) else offs[j-1]+len(tokens[j-1])))
            i=j
            continue
        i+=1
    return text, spans

**Build gold spans for validation & test**

In [20]:
gold_val = [bio_to_spans(t, y)[1] for t,y in zip(df_val.tokens, df_val.ner_tags)]
gold_test = [bio_to_spans(t, y)[1] for t,y in zip(df_test.tokens, df_test.ner_tags)]

**Preview one sample**

In [21]:
sample_i = 0
df_val.text.iloc[sample_i], gold_val[sample_i][:5]

('CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTER INNINGS VICTORY .',
 [('ORG', 10, 24)])

**Save preprocessed splits**

In [22]:
df_train.to_parquet("conll_train.parquet", index=False)
df_val.to_parquet("conll_val.parquet", index=False)
df_test.to_parquet("conll_test.parquet", index=False)

# **3. Pipelines (Rule-based + spaCy)**

**Install**

In [23]:
!pip -q install spacy spacy-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.8/758.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Download spaCy models**

In [24]:
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 38.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 735.6/735.6 kB 3.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Load model-based pipelines**

In [25]:
import spacy
from spacy.pipeline import EntityRuler
nlp_sm = spacy.load("en_core_web_sm")
nlp_trf = spacy.load("en_core_web_trf")

**Build rule-based baseline**

In [26]:
nlp_rule = spacy.blank("en")
ruler = nlp_rule.add_pipe("entity_ruler")
ruler.add_patterns([{"label":"ORG","pattern":"United Nations"},{"label":"ORG","pattern":"European Union"},{"label":"GPE","pattern":"United States"},{"label":"GPE","pattern":"Germany"},{"label":"PERSON","pattern":[{"IS_TITLE":True},{"IS_TITLE":True}]},{"label":"MISC","pattern":"Premier League"}])

# **4. Evaluation**

**Prep: texts & tqdm**

In [27]:
from tqdm import tqdm
texts_val, texts_test = df_val.text.tolist(), df_test.text.tolist()

**BIO → char spans**

In [28]:
def gold_excl(tok,tg):
    spans=[]
    off=[0]
    [off.append(off[-1]+len(w)+1) for w in tok]
    i=0
    while i<len(tg):
        if tg[i].startswith("B-"):
            lab=tg[i][2:]
            j=i+1
            while j<len(tg) and tg[j]==f"I-{lab}": j+=1
            spans.append((lab, off[i], off[j-1]+len(tok[j-1])))
            i=j
            continue
        i+=1
    return spans

**Build gold spans**

In [29]:
gold_val = [gold_excl(t,y) for t,y in zip(df_val.tokens, df_val.ner_tags)]
gold_test = [gold_excl(t,y) for t,y in zip(df_test.tokens, df_test.ner_tags)]

**Predict spans helper**

In [30]:
def pred(nlp, txts): return [[(e.label_,e.start_char,e.end_char) for e in nlp(t).ents] for t in tqdm(txts)]

**Run predictions (validation)**

In [31]:
pred_rule_val = pred(nlp_rule, texts_val)
pred_sm_val   = pred(nlp_sm,   texts_val)
pred_trf_val  = pred(nlp_trf,  texts_val)

100%|██████████| 3250/3250 [07:30<00:00,  7.21it/s]


**P/R/F1 function**

In [33]:
def prf1(P,G):
    tp=fp=fn=0
    for p,g in zip(P,G): ps=set(p)
    gs=set(g)
    tp+=len(ps&gs)
    fp+=len(ps-gs)
    fn+=len(gs-ps)
    P_=tp/(tp+fp+1e-9)
    R_=tp/(tp+fn+1e-9)
    F_=2*P_*R_/(P_+R_+1e-9)
    return {"precision":P_,"recall":R_,"f1":F_,"tp":tp,"fp":fp,"fn":fn}

**Compute metrics (validation)**

In [34]:
m_rule_val = prf1(pred_rule_val, gold_val)
m_sm_val   = prf1(pred_sm_val,   gold_val)
m_trf_val  = prf1(pred_trf_val,  gold_val)

**Show validation table**

In [40]:
import pandas as pd
val_df = pd.DataFrame([["val","rule",m_rule_val['precision'],m_rule_val['recall'],m_rule_val['f1'],m_rule_val['tp'],m_rule_val['fp'],m_rule_val['fn']],
                       ["val","spacy_sm",m_sm_val['precision'],m_sm_val['recall'],m_sm_val['f1'],m_sm_val['tp'],m_sm_val['fp'],m_sm_val['fn']],
                       ["val","spacy_trf",m_trf_val['precision'],m_trf_val['recall'],m_trf_val['f1'],m_trf_val['tp'],m_trf_val['fp'],m_trf_val['fn']]],
                      columns=["split","model","precision","recall","f1","tp","fp","fn"])
val_df

,split,model,precision,recall,f1,tp,fp,fn
0,val,rule,0.000000,0.000000,0.000000,0,2235,1
1,val,spacy_sm,0.050308,0.997778,0.095787,449,8476,1
2,val,spacy_trf,0.064840,0.998408,0.121771,627,9043,1


**Run predictions (test)**

In [36]:
pred_rule_test = pred(nlp_rule, texts_test)
pred_sm_test   = pred(nlp_sm,   texts_test)
pred_trf_test  = pred(nlp_trf,  texts_test)

100%|██████████| 3453/3453 [07:28<00:00,  7.69it/s]


**Compute metrics (test)**

In [37]:
m_rule_test = prf1(pred_rule_test, gold_test)
m_sm_test   = prf1(pred_sm_test,   gold_test)
m_trf_test  = prf1(pred_trf_test,  gold_test)

**Show test table**

In [39]:
test_df = pd.DataFrame([["test","rule",m_rule_test['precision'],m_rule_test['recall'],m_rule_test['f1'],m_rule_test['tp'],m_rule_test['fp'],m_rule_test['fn']],
                        ["test","spacy_sm",m_sm_test['precision'],m_sm_test['recall'],m_sm_test['f1'],m_sm_test['tp'],m_sm_test['fp'],m_sm_test['fn']],
                        ["test","spacy_trf",m_trf_test['precision'],m_trf_test['recall'],m_trf_test['f1'],m_trf_test['tp'],m_trf_test['fp'],m_trf_test['fn']]],
                       columns=["split","model","precision","recall","f1","tp","fp","fn"])
test_df

,split,model,precision,recall,f1,tp,fp,fn
0,test,rule,0.000455,0.200000,0.000908,1,2196,4
1,test,spacy_sm,0.063569,0.992780,0.119487,550,8102,4
2,test,spacy_trf,0.084390,0.996063,0.155597,759,8235,3


# **5. Visualization (displaCy + HTML saves)**

**Import + choose a sample**

In [41]:
from spacy import displacy
sample_i = 0
sample_txt = df_val.text.iloc[sample_i]

**Render - spaCy small**

In [42]:
displacy.render(nlp_sm(sample_txt), style="ent", jupyter=True, options={"compact":True})

/usr/local/lib/python3.11/dist-packages/spacy/displacy/__init__.py:213: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


**Render - spaCy transformer**

In [43]:
displacy.render(nlp_trf(sample_txt), style="ent", jupyter=True, options={"compact":True})

**Render - Rule-based**

In [44]:
displacy.render(nlp_rule(sample_txt), style="ent", jupyter=True, options={"compact":True})

/usr/local/lib/python3.11/dist-packages/spacy/displacy/__init__.py:213: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


## **Save HTML files**

In [46]:
from spacy import displacy
to_html = lambda d: displacy.render(d, style="ent", page=True, jupyter=False, options={"compact":True})

**Save - spaCy small**

In [47]:
open("viz_sm.html","w",encoding="utf-8").write(to_html(nlp_sm(sample_txt)))

505

**Save - transformer**

In [48]:
open("viz_trf.html","w",encoding="utf-8").write(to_html(nlp_trf(sample_txt)))

505

**Save - rule-based**

In [49]:
open("viz_rule.html","w",encoding="utf-8").write(to_html(nlp_rule(sample_txt)))

/usr/local/lib/python3.11/dist-packages/spacy/displacy/__init__.py:213: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


505

# **6. Save Outputs (tables + sample predictions)**

**Combine & save metrics**

In [52]:
summary = pd.concat([val_df, test_df], ignore_index=True)
summary.to_csv("ner_results_summary.csv", index=False)
summary

,split,model,precision,recall,f1,tp,fp,fn
0,val,rule,0.000000,0.000000,0.000000,0,2235,1
1,val,spacy_sm,0.050308,0.997778,0.095787,449,8476,1
2,val,spacy_trf,0.064840,0.998408,0.121771,627,9043,1
3,test,rule,0.000455,0.200000,0.000908,1,2196,4
4,test,spacy_sm,0.063569,0.992780,0.119487,550,8102,4
5,test,spacy_trf,0.084390,0.996063,0.155597,759,8235,3


**serialize spans**

In [53]:
to_js = lambda spans: [{"label":l,"start":s,"end":e} for (l,s,e) in spans]
gold_js = [to_js(g) for g in gold_val[:50]]

**Build sample predictions DF**

In [54]:
pred_sm_js  = [to_js(p) for p in pred_sm_val[:50]]
pred_trf_js = [to_js(p) for p in pred_trf_val[:50]]
pred_rule_js= [to_js(p) for p in pred_rule_val[:50]]

**Export sample predictions**

In [57]:
export = pd.DataFrame({"text":df_val.text.iloc[:50],"gold":gold_js,"pred_sm":pred_sm_js,"pred_trf":pred_trf_js,"pred_rule":pred_rule_js})
export.to_csv("ner_sample_predictions.csv", index=False)
export.head()

,text,gold,pred_sm,pred_trf,pred_rule
0,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'label': 'ORG', 'start': 10, 'end': 24}]",[],[],[]
1,LONDON 1996-08-30,"[{'label': 'LOC', 'start': 0, 'end': 6}]","[{'label': 'GPE', 'start': 0, 'end': 6}, {'lab...","[{'label': 'GPE', 'start': 0, 'end': 6}, {'lab...",[]
2,West Indian all-rounder Phil Simmons took four...,"[{'label': 'MISC', 'start': 0, 'end': 11}, {'l...","[{'label': 'NORP', 'start': 0, 'end': 11}, {'l...","[{'label': 'NORP', 'start': 0, 'end': 11}, {'l...","[{'label': 'PERSON', 'start': 0, 'end': 11}, {..."
3,"Their stay on top , though , may be short-live...","[{'label': 'ORG', 'start': 64, 'end': 69}, {'l...","[{'label': 'PERSON', 'start': 72, 'end': 82}, ...","[{'label': 'ORG', 'start': 64, 'end': 69}, {'l...",[]
4,After bowling Somerset out for 83 on the openi...,"[{'label': 'ORG', 'start': 14, 'end': 22}, {'l...","[{'label': 'GPE', 'start': 14, 'end': 22}, {'l...","[{'label': 'ORG', 'start': 14, 'end': 22}, {'l...","[{'label': 'PERSON', 'start': 60, 'end': 70}, ..."
